In [ ]:
# Système RAG (Retrieval-Augmented Generation) avec Langchain et Hugging Face

# Étape 1: Installation des bibliothèques nécessaires
!pip install -q langchain torch transformers sentence-transformers datasets faiss-cpu langchain-community

# Étape 2: Importation des modules
from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# Étape 3: Chargement du dataset
print("Loading dataset...")
loader = HuggingFaceDatasetLoader("databricks/databricks-dolly-15k", page_content_column="context")
data = loader.load()
print(f"Dataset loaded: {len(data)} documents")

# Étape 4: Découpage des documents en chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
splits = text_splitter.split_documents(data)
print(f"Created {len(splits)} chunks")

# Étape 5: Création des embeddings et du vector store
print("Creating embeddings and vector store...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created successfully")

# Étape 6: Préparation du modèle LLM pour le Question Answering
print("Loading Question Answering model...")
model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
qa_pipeline = pipeline("question-answering", model=model, tokenizer=tokenizer)
llm = HuggingFacePipeline(pipeline=qa_pipeline)
print("Model loaded successfully")

# Étape 7: Construction de la chaîne RetrievalQA
print("Building RetrievalQA chain...")
retriever = vectorstore.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)
print("RAG system ready!")

# Étape 8: Test du système avec une requête
print("\n=== Testing RAG System ===")
query = "What is cheesemaking?"
print(f"Query: {query}")

result = qa_chain({"query": query})
print(f"\nAnswer: {result['result']}")
print(f"\nSource documents retrieved: {len(result['source_documents'])}")
for i, doc in enumerate(result['source_documents'][:2]):
    print(f"\nSource {i+1}: {doc.page_content[:200]}...")